# 05 · Gene Set Enrichment Analysis (GSEA)
**Input :** `data/processed/dea_results.csv`  
**Output:** `results/tables/gsea_go_bp.csv` · `gsea_go_mf.csv` · `gsea_go_cc.csv`
· `gsea_kegg.csv` · `results/figures/gsea_*.png`

Runs ranked GSEA via clusterProfiler (R) across GO-BP, GO-MF, GO-CC, and KEGG.
Highlights four HCC-relevant themes: lipid metabolism, glycolysis, PI3K-AKT/Wnt,
and immune regulation.

**Requires:** R packages `clusterProfiler`, `org.Hs.eg.db`, `enrichplot`.  
Install with: `Rscript env/r_packages.R`

**Run order:** 04 → **05** → P1


In [1]:
import sys
from pathlib import Path

def _find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "paths.py").exists():
            return p
    raise FileNotFoundError("paths.py not found — run: python scripts/data_download.py")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))

from paths import REPO_ROOT, RAW_DIR, PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
print(f"Repo root : {REPO_ROOT}")
print(f"Raw data  : {RAW_DIR}")

Repo root : C:\Users\shoko\OneDrive\Desktop\project\HCC_DD
Raw data  : C:\Users\shoko\OneDrive\Desktop\project\HCC_DD\data\raw


In [2]:
from utils import prepare_ranked_list, run_gsea_r, print_gsea_summary
print("Imports OK")

Imports OK


## 1 · R environment (run once per session)

In [3]:
import os, sys

# ── R environment — update R_HOME if needed ───────────────────────────────────
# macOS  : /Library/Frameworks/R.framework/Resources
# Linux  : /usr/lib/R
# Windows: r"C:\Program Files\R\R-4.5.3"
R_HOME = r"C:\Program Files\R\R-4.5.3"
if R_HOME:
    os.environ["R_HOME"] = R_HOME
    if sys.platform == "win32":
        os.environ["PATH"] = os.path.join(R_HOME,"bin","x64") + ";" + os.environ["PATH"]

%load_ext rpy2.ipython
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import default_converter
import anndata2ri
print("R environment ready")

R environment ready


## 2 · Install GSEA R packages (safe to re-run)

In [4]:
%%R
pkgs <- c("clusterProfiler","org.Hs.eg.db","enrichplot","DOSE","ggplot2")
if (!requireNamespace("BiocManager", quietly=TRUE))
    install.packages("BiocManager", repos="https://cloud.r-project.org")
for (pkg in pkgs)
    if (!requireNamespace(pkg, quietly=TRUE))
        BiocManager::install(pkg, ask=FALSE, update=FALSE)
suppressPackageStartupMessages({
    library(clusterProfiler); library(org.Hs.eg.db)
    library(enrichplot);      library(ggplot2)
})
cat("GSEA packages ready\n")

GSEA packages ready




In addition: Warning message:

   File C:\Users\shoko\OneDrive\Documents/.Renviron contains invalid line(s)
tools40"
   They were ignored
 


## 3 · Prepare ranked gene list

In [5]:
ranked = prepare_ranked_list(PROC_DIR / "dea_results.csv", PROC_DIR)
ranked.head(10)

Ranked list: 1385 genes → C:\Users\shoko\OneDrive\Desktop\project\HCC_DD\data\processed\ranked_genes_log2fc.tsv


,gene,log2FC
1361,XIST,31.180847
880,OLFM4,28.968952
29,ADH1C,28.503433
1235,TFF2,28.247526
265,CEACAM6,28.181560
777,MIA,28.101700
1162,SMIM22,27.783546
101,AQP5,27.590366
508,GABRP,27.233480
337,CTSE,9.171602


## 4 · Run GSEA in R

In [6]:
run_gsea_r(ro, PROC_DIR, FIGURES_DIR, TABLES_DIR)

R callback write-console: 'select()' returned 1:1 mapping between keys and columns
  


Genes mapped to Entrez: 1314 / 1385
Running GO-BP...
Running GO-MF...
Running GO-CC...
Running KEGG...


R callback write-console: Reading KEGG annotation online: "https://rest.kegg.jp/link/hsa/pathway"...
  
R callback write-console: Reading KEGG annotation online: "https://rest.kegg.jp/list/pathway/hsa"...
  



GO-BP : 120 terms
GO-MF : 10 terms
GO-CC : 16 terms
KEGG  : 2 pathways
Saved: C:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\figures/gsea_go_biological_process.png
Saved: C:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\figures/gsea_go_molecular_function.png
Saved: C:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\figures/gsea_go_cellular_component.png
Saved: C:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\figures/gsea_kegg_pathways.png


R callback write-console: Picking joint bandwidth of 2.99
  


Saved: C:\Users\shoko\OneDrive\Desktop\project\HCC_DD\results\figures/gsea_ridgeplot_bp.png
Saved theme: Lipid_metabolism
Saved theme: Immune_regulation
CSV tables saved.


R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In bitr(rnk$SYMBOL, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = org.Hs.eg.db) :  
R callback write-console: 
   
R callback write-console:  5.13% of input gene IDs are fail to map...
  
R callback write-console: 2:   
R callback write-console: In fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize,  :  
R callback write-console: 
   
R callback write-console:  For some of the pathways the P-values were likely overestimated. For such pathways log2err is set to NA.
  
R callback write-console: 3:   
R callback write-console: In fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize,  :  
R callback write-console: 
   
R callback write-console:  There were 2 pathways for which P-values were not calculated properly due to unbalanced (positive and negative) gene-level statistic values. For such pathways pval


✓ GSEA complete


## 5 · Results summary

In [7]:
print_gsea_summary(TABLES_DIR)

── GSEA results summary ──

  GO-BP: 120 enriched terms
                  Description       NES  p.adjust
establishment of localization -1.319186  0.041753
                    transport -1.335020  0.041753
              immune response -1.373900  0.046330

  GO-MF: 10 enriched terms
               Description       NES  p.adjust
    small molecule binding -1.347953  0.029355
signaling receptor binding -1.603512  0.013556
             lipid binding -1.869436  0.000858

  GO-CC: 16 enriched terms
                             Description       NES  p.adjust
                   extracellular exosome -1.396657   0.01603
                 extracellular organelle -1.399694   0.01603
extracellular membrane-bounded organelle -1.399694   0.01603

  KEGG: 2 enriched terms
           Description       NES  p.adjust
PPAR signaling pathway -2.011785  0.002554
Cholesterol metabolism -2.097698  0.001062

── HCC-relevant pathway hits (GO-BP) ──

  Lipid metabolism: 29 terms
    response to lipid -1.52959